# Dual-Core Tokenizer-Only Walkthrough

This notebook matches the current phase-1 scope:

- Twi only
- tokenizer-only experiments
- one benchmark experiment = one JSON output

The flow is:

1. install dependencies
2. download and normalize Twi data
3. train `asr`, `tts`, and `mixed` tokenizers
4. run one unified fertility experiment
5. inspect the result JSON


In [ ]:
%pip install -e ".[dev]"

## Download Twi data

This writes normalized JSONL files into `data/`.


In [ ]:
!python scripts/download.py --output-dir data

## Train tokenizer variants

Each run writes one tokenizer artifact plus one stats JSON.


In [ ]:
!python scripts/train_bpe.py --inputs data/aka_asr_train.jsonl --output models/asr_tokenizer.json --name asr --vocab-size 8000

In [ ]:
!python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts --vocab-size 8000

In [ ]:
!python scripts/train_bpe.py --inputs data/aka_asr_train.jsonl data/pristine_twi_train.jsonl --output models/mixed_tokenizer.json --name mixed --vocab-size 8000

## Run one unified fertility experiment

This writes one comparison JSON under `results/`.


In [ ]:
!python scripts/benchmark_fertility.py --experiment-id tokenizer_fertility_experiment_001 --control-tokenizer gpt2 --asr-tokenizer models/asr_tokenizer.json --tts-tokenizer models/tts_tokenizer.json --mixed-tokenizer models/mixed_tokenizer.json --asr-test-file data/aka_asr_test.jsonl --tts-test-file data/pristine_twi_test.jsonl --output results/tokenizer_fertility_experiment_001.json

## Inspect the unified result JSON

This is the main output for one complete experiment run.


In [ ]:
import json
from pathlib import Path

result_path = Path("results/tokenizer_fertility_experiment_001.json")
payload = json.loads(result_path.read_text(encoding="utf-8"))
payload["summary"]

In [ ]:
for tokenizer_name, tokenizer_results in payload["results"].items():
    print(tokenizer_name)
    print("  ASR fertility:", tokenizer_results["asr_test"]["fertility"])
    print("  TTS fertility:", tokenizer_results["tts_test"]["fertility"])
    print()